# Knee: fold-0 backbone / training-length screen (ConvNeXt-Small vs ConvNeXt-Tiny 20 epochs)
Two jobs run **in parallel in this one session**, one per GPU (one session = one quota charge; two separate notebooks would be charged twice):
- GPU 0: `cnxS` = `convnext_small.in12k_ft_in1k`, 12 epochs, bs 2 x accum 4 (same effective batch 8, lower memory)
- GPU 1: `cnxT_e20` = `convnext_tiny.in12k_ft_in1k`, 20 epochs, bs 4 x accum 2

Both: head `tokens`, K=1, 160 px cache, `--only_folds 0`, `--train_slices 16`, seed 42, same labels / recipe as the ConvNeXt-T 5-fold.
Compared on fold 0 against the ConvNeXt-T 5-fold OOF (seed 42 and seed 43), on the FIXED old labels (confident y >= 0.9 or <= 0.1), ranks within fold 0.
Inputs: competition data is NOT needed. Attach: `knee-mri-pipeline` dataset (your patched v6_3), the caching output (cache + `labels_soft.csv`), the baseline training output (`oof_all.csv`), the ConvNeXt 5-fold outputs (`oof_v6convnext.csv`, `oof_v6convnext2.csv`).
GPU T4 x2, internet ON (pretrained weights download), Save & Run All. Expected wall time ~3-4 h (estimate: watch the epoch times printed every 10 min; if `cnxS` projects over ~25 min/epoch, stop it and keep `cnxT_e20`).

In [1]:
import os, sys, json, subprocess, time, threading
from pathlib import Path
import numpy as np, pandas as pd

W = Path(os.environ.get("KNEE_WORK", "/kaggle/working"))
CODE   = Path(os.environ.get("KNEE_CODE",   "/kaggle/input/datasets/theokorir/knee-mri-pipeline/knee_pipeline"))
CACHE  = Path(os.environ.get("KNEE_CACHE",  "/kaggle/input/notebooks/theokorir/knee-mri-diagnosis-caching/cache"))
LABELS = Path(os.environ.get("KNEE_LABELS", "/kaggle/input/notebooks/theokorir/knee-mri-diagnosis-caching/labels_soft.csv"))
for n, p in dict(CODE=CODE, CACHE=CACHE, LABELS=LABELS).items(): assert p.exists(), f"{n} not found: {p}"
assert (CODE / "train.py").exists() and (CODE / "geom.py").exists(), "dataset does not hold knee_pipeline_v6"

# arm name -> (GPU, backbone, epochs, bs, accum). Edit here to try other backbones.
ARMS = {
    "cnxS":     (0, "convnext_small.in12k_ft_in1k", 12, 2, 4),
    "cnxT_e20": (1, "convnext_tiny.in12k_ft_in1k",  20, 4, 2),
}
COMMON = ["--cache_dir", str(CACHE), "--labels", str(LABELS), "--pretrained", "--workers", "2",
          "--train_slices", "16", "--head", "tokens", "--K", "1", "--only_folds", "0", "--seed", "42"]
print({k: v for k, v in ARMS.items()})

{'cnxS': (0, 'convnext_small.in12k_ft_in1k', 12, 2, 4), 'cnxT_e20': (1, 'convnext_tiny.in12k_ft_in1k', 20, 4, 2)}


In [2]:
# fail fast: make sure every backbone builds AND its pretrained weights download (internet must be ON)
import timm, torch
for name, (g, bb, ep, bs, ac) in ARMS.items():
    t = time.time(); m = timm.create_model(bb, pretrained=True, num_classes=0)
    print(f"{name}: {bb} pretrained OK | {sum(p.numel() for p in m.parameters())/1e6:.1f}M params | {time.time()-t:.0f}s")
    del m
# code smoke test (CPU, ~1 min)
for t in ("tests/test_geom.py", "tests/smoke_v6.py"):
    r = subprocess.run([sys.executable, str(CODE / t)], capture_output=True, text=True, cwd=str(CODE))
    print(f"== {t} (exit {r.returncode})")
    assert r.returncode == 0, f"{t} failed:\n{r.stderr[-1500:]}"

model.safetensors: reconstructing file:   0%|          |  0.00B /  201MB            

model.safetensors: downloading bytes:           |  0.00B            

cnxS: convnext_small.in12k_ft_in1k pretrained OK | 49.5M params | 3s


model.safetensors: reconstructing file:   0%|          |  0.00B /  114MB            

model.safetensors: downloading bytes:           |  0.00B            

cnxT_e20: convnext_tiny.in12k_ft_in1k pretrained OK | 27.8M params | 2s
== tests/test_geom.py (exit 0)
== tests/smoke_v6.py (exit 0)


In [3]:
LOGS, RESULT = {}, {}
def run(name, gpu, bb, ep, bs, ac):
    out = W / f"screen_{name}"
    if (out / "oof.csv").exists():
        print(f"[{name}] oof.csv exists, skipped"); RESULT[name] = 0; return
    cmd = [sys.executable, str(CODE / "train.py")] + COMMON + ["--backbone", bb, "--epochs", str(ep), "--bs", str(bs), "--accum", str(ac), "--out", str(out)]
    LOGS[name] = W / f"screen_{name}.log"
    print(f"[{name}] GPU {gpu}: {' '.join(cmd)}", flush=True)
    with open(LOGS[name], "w") as fh:
        p = subprocess.Popen(cmd, stdout=fh, stderr=subprocess.STDOUT, env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu)), cwd=str(CODE))
        RESULT[name] = p.wait()
    print(f"[{name}] exit code {RESULT[name]}", flush=True)
threads = [threading.Thread(target=run, args=(n, *a)) for n, a in ARMS.items()]
for th in threads: th.start()
t0 = time.time()
while any(th.is_alive() for th in threads):
    time.sleep(600)
    last = {}
    for n, f in LOGS.items():
        ls = f.read_text().splitlines() if f.exists() else []
        ep = [l for l in ls if " ep " in l]; it = [l for l in ls if " it " in l]
        last[n] = (ep[-1] if ep else "(no epoch yet)") + " || " + (it[-1] if it else "")
    print(f"[{(time.time()-t0)/3600:.2f} h] {json.dumps(last, indent=1)}", flush=True)
for th in threads: th.join()
print("all jobs finished | results:", RESULT)
assert all(v == 0 for v in RESULT.values()), "a job failed: see screen_*.log"

[cnxS] GPU 0: /usr/bin/python3 /kaggle/input/datasets/theokorir/knee-mri-pipeline/knee_pipeline/train.py --cache_dir /kaggle/input/notebooks/theokorir/knee-mri-diagnosis-caching/cache --labels /kaggle/input/notebooks/theokorir/knee-mri-diagnosis-caching/labels_soft.csv --pretrained --workers 2 --train_slices 16 --head tokens --K 1 --only_folds 0 --seed 42 --backbone convnext_small.in12k_ft_in1k --epochs 12 --bs 2 --accum 4 --out /kaggle/working/screen_cnxS
[cnxT_e20] GPU 1: /usr/bin/python3 /kaggle/input/datasets/theokorir/knee-mri-pipeline/knee_pipeline/train.py --cache_dir /kaggle/input/notebooks/theokorir/knee-mri-diagnosis-caching/cache --labels /kaggle/input/notebooks/theokorir/knee-mri-diagnosis-caching/labels_soft.csv --pretrained --workers 2 --train_slices 16 --head tokens --K 1 --only_folds 0 --seed 42 --backbone convnext_tiny.in12k_ft_in1k --epochs 20 --bs 4 --accum 2 --out /kaggle/working/screen_cnxT_e20
[0.17 h] {
 "cnxS": "(no epoch yet) || [f0] it 1450/1762 | data-wait 0.

In [4]:
# ---- analysis: fold 0, fixed old labels, ranks within fold 0 ----
from scipy.stats import rankdata
ROOT = Path(os.environ.get("KAGGLE_INPUT_ROOT", "/kaggle/input"))
LAB = ["ACL","MCL","Medial Meniscus","Lateral Meniscus","Medial OA","Lateral OA","PF OA","Effusion","Synovitis","Baker's","Contusion","Fracture"]
KEY4 = LAB[:4]
want = {"baseline": "oof_all.csv", "convnext": "oof_v6convnext.csv", "convnext_s43": "oof_v6convnext2.csv"}
SKIP = {"cache", "train_series", "test_series", "train", "test", "__pycache__", ".git"}
paths = {}
for d, dirs, fs in os.walk(ROOT):
    dirs[:] = [x for x in dirs if x not in SKIP]
    for f in fs:
        for k, v in want.items():
            if f == v: paths.setdefault(k, []).append(Path(d) / f)
for k, v in paths.items(): print(k, v)
for k in want:
    if k in paths and len(paths[k]) > 1: raise SystemExit(f"{k}: several files found, set paths['{k}'] = [the one you want]")
need = ["baseline", "convnext"]
assert all(k in paths for k in need), f"missing inputs: {[k for k in need if k not in paths]}"
ref = pd.read_csv(paths["convnext"][0])
assert "fold" in ref.columns and "StudyInstanceUID" in ref.columns
f0 = ref.fold.values == 0; ids0 = ref.StudyInstanceUID[f0].tolist(); n0 = len(ids0); print("fold-0 studies:", n0)
def load(path):
    d = pd.read_csv(path).set_index("StudyInstanceUID").loc[ids0]
    return np.stack([d[f"p_{l}"].values for l in LAB], 1).astype(float), np.stack([d[f"y_{l}"].values for l in LAB], 1).astype(float)
P, Y = {}, None
for k in want:
    if k in paths: P[k], y = load(paths[k][0]); Y = y if Y is None else Y
for name in ARMS:
    d = pd.read_csv(W / f"screen_{name}" / "oof.csv").set_index("StudyInstanceUID")
    d = d.loc[ids0]; assert d[[f"p_{l}" for l in LAB]].notna().all().all(), f"{name}: NaN predictions on fold-0 studies"
    P[name] = np.stack([d[f"p_{l}"].values for l in LAB], 1).astype(float)
    assert np.allclose(np.stack([d[f"y_{l}"].values for l in LAB], 1), Y, atol=1e-6), f"{name}: y_ differs from the fixed old labels"
RK = {n: np.apply_along_axis(rankdata, 0, p) / n0 for n, p in P.items()}
def _auc(yb, s):
    n1 = int(yb.sum()); n0_ = len(yb) - n1
    if n1 < 5 or n0_ < 5: return np.nan
    r = rankdata(s); return (r[yb == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0_)
def aucs(R, rows=None):
    out = np.full(12, np.nan)
    for j in range(12):
        y = Y[:, j] if rows is None else Y[rows, j]; s = R[:, j] if rows is None else R[rows, j]
        m = (y >= .9) | (y <= .1)
        if m.sum() > 20: out[j] = _auc((y[m] >= .9).astype(int), s[m])
    return out
macro = lambda a: np.nanmean(a); key4 = lambda a: np.nanmean(a[:4])
cols = dict(P); 
base_ens = ["convnext"] + (["convnext_s43"] if "convnext_s43" in RK else [])
for n in ARMS:
    cols[f"{n} + convnext"] = (RK[n] + RK["convnext"]) / 2
    cols[f"convnext + convnext_s43 + {n}"] = None if "convnext_s43" not in RK else (RK["convnext"] + RK["convnext_s43"] + RK[n]) / 3
if "convnext_s43" in RK: cols["ens convnext x2 seeds"] = (RK["convnext"] + RK["convnext_s43"]) / 2
T = pd.DataFrame({n: aucs(RK[n] if n in RK else v) for n, v in cols.items() if (n in RK or v is not None)}, index=LAB)
T.loc["MACRO"] = T.loc[LAB].mean(); T.loc["KEY-4"] = T.loc[KEY4].mean()
print("Fold 0 (AUC on confident old labels, ranks within fold 0):\n", T.round(3).to_string())
if "convnext_s43" in RK:
    a, b = aucs(RK["convnext"]), aucs(RK["convnext_s43"])
    print(f"\nConvNeXt-T seed noise on this fold: macro {abs(macro(a)-macro(b)):.4f}, KEY-4 {abs(key4(a)-key4(b)):.4f}")
rng = np.random.default_rng(0); boots = [rng.integers(0, n0, n0) for _ in range(300)]
print("\nVERDICTS (reference = ConvNeXt-T 12 ep seed 42; rule: > +0.015 macro = run the 5-fold; within 0.005 = diversity candidate; else drop)")
for n in ARMS:
    d = macro(aucs(RK[n])) - macro(aucs(RK["convnext"])); dk = key4(aucs(RK[n])) - key4(aucs(RK["convnext"]))
    bd = [macro(aucs(RK[n], b)) - macro(aucs(RK["convnext"], b)) for b in boots]
    ens_gain = ""
    if "convnext_s43" in RK:
        e0 = (RK["convnext"] + RK["convnext_s43"]) / 2; e1 = (RK["convnext"] + RK["convnext_s43"] + RK[n]) / 3
        ens_gain = f" | adding it to the 2-seed ConvNeXt ensemble: {macro(aucs(e1))-macro(aucs(e0)):+.4f} macro"
    v = "RUN 5-FOLD" if d > 0.015 else ("diversity candidate" if d > -0.005 else "drop")
    print(f"  {n}: macro {d:+.4f} (90% CI {np.percentile(bd,5):+.4f}..{np.percentile(bd,95):+.4f}), KEY-4 {dk:+.4f}{ens_gain}  ->  {v}")

baseline [PosixPath('/kaggle/input/notebooks/theokorir/knee-mri-diagnosis-training/oof_all.csv')]
convnext [PosixPath('/kaggle/input/notebooks/theokorir/knee-v6-convnext-5fold/oof_v6convnext.csv')]
fold-0 studies: 882
Fold 0 (AUC on confident old labels, ranks within fold 0):
                   baseline  convnext   cnxS  cnxT_e20  cnxS + convnext  cnxT_e20 + convnext
ACL                  0.833     0.892  0.870     0.899            0.888                0.899
MCL                  0.713     0.805  0.772     0.812            0.794                0.813
Medial Meniscus      0.757     0.870  0.869     0.890            0.873                0.882
Lateral Meniscus     0.715     0.776  0.750     0.818            0.765                0.800
Medial OA            0.861     0.874  0.875     0.879            0.876                0.878
Lateral OA           0.749     0.767  0.753     0.784            0.763                0.778
PF OA                0.722     0.740  0.754     0.731            0.750        